# 3.30 — The Kernel Trick & Kernel Methods

A kernel method learns with similarities instead of hand-building every feature coordinate. In this lesson, we will start from the identity $K(x,z)=\phi(x)^\top\phi(z)$, verify concrete kernels by arithmetic, and then use kernel Gram matrices to make small predictions while keeping the empirical-risk, cost, validation-gap, and regularization logic visible.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build kernel methods one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is shown so the kernel trick does not feel like magic. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, distances, linear algebra, and numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for the toy data.

### 1. A kernel is an inner product after a feature map

The kernel trick starts with an ordinary inner product, but not necessarily in the original input coordinates. A feature map $\phi(x)$ may add nonlinear coordinates such as squares and cross terms. If we can compute $K(x,z)=\phi(x)^\top\phi(z)$ directly, then algorithms that only need dot products can act as if they saw the expanded features without explicitly storing them.

In [ ]:
x_w = np.array([2.0, 1.0])  # one 2-D point.
z_w = np.array([1.0, 3.0])  # another 2-D point.
phi_x_w = np.array([x_w[0]**2, np.sqrt(2)*x_w[0]*x_w[1], x_w[1]**2])  # degree-2 map.
phi_z_w = np.array([z_w[0]**2, np.sqrt(2)*z_w[0]*z_w[1], z_w[1]**2])  # same map for z.
print("phi(x):", np.round(phi_x_w, 3))
print("phi(z):", np.round(phi_z_w, 3))

▶ What you'll see: two 3-D feature vectors built from two original coordinates.

In [ ]:
explicit_inner_w = float(phi_x_w @ phi_z_w)  # inner product after expansion.
direct_kernel_w = float((x_w @ z_w) ** 2)  # polynomial kernel with degree 2 and no offset.
print("explicit phi(x)·phi(z):", round(explicit_inner_w, 3))
print("direct (x·z)^2:", round(direct_kernel_w, 3))
assert round(explicit_inner_w, 3) == 25.0
assert round(direct_kernel_w, 3) == 25.0

▶ What you'll see: both calculations give 25, so the direct kernel exactly matches the expanded dot product.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["x0²", "√2 x0x1", "x1²"], phi_x_w, color="teal")
plt.title("1: explicit quadratic features for x")
plt.ylabel("feature value")
plt.show()

▶ What you'll see: the cross-term coordinate is large because both original coordinates participate.

*Why it's done this way:* the expanded map contains every quadratic interaction needed by a degree-2 linear model, while the kernel computes their inner product from the original dot product. The formula is not an approximation here; the $\sqrt{2}$ makes the cross term counted with the exact coefficient that appears when $(x^\top z)^2$ is expanded.

### 2. Polynomial kernels add nonlinear boundaries without drawing all features

A degree-$d$ polynomial kernel $K(x,z)=(x^\top z+c)^d$ behaves like a dot product over all monomials up to degree $d$ when $c>0$. Larger degree means more flexible interactions; the offset $c$ keeps lower-order terms alive so the model can still use simple linear evidence.

In [ ]:
points_w = np.array([[-2.0], [-1.0], [0.0], [1.0], [2.0]])  # one-dimensional inputs for inspection.
center_w = np.array([[1.0]])  # compare every point to z=1.
linear_sim_w = points_w @ center_w.T  # ordinary dot products.
poly_sim_w = (points_w @ center_w.T + 1.0) ** 3  # cubic polynomial kernel with offset.
print("linear similarities:", linear_sim_w.ravel())
print("cubic kernel similarities:", poly_sim_w.ravel())

▶ What you'll see: the cubic kernel changes both scale and curvature; points near the center become much more separated.

In [ ]:
grid_w = np.linspace(-2.5, 2.5, 101)[:, None]  # smooth 1-D grid.
sim_curve_w = (grid_w @ center_w.T + 1.0).ravel() ** 3  # K(x, 1) for each grid point.
plt.figure(figsize=(4.4, 3))
plt.plot(grid_w.ravel(), sim_curve_w, color="purple")
plt.axvline(1.0, color="black", linestyle="--")
plt.title("2: cubic polynomial similarity to z=1")
plt.xlabel("x")
plt.ylabel("K(x, 1)")
plt.show()

▶ What you'll see: similarity is not a straight line; the cubic feature space bends the comparison.

*Why it's done this way:* kernel algorithms combine training examples through similarities. If the similarity itself already contains powers and interactions, a linear combination of kernels can represent curved decision rules while the optimization still manipulates dot products.

### 3. The RBF kernel measures local similarity with a bandwidth

The radial basis function kernel is $K(x,z)=\exp(-\gamma\lVert x-z\rVert^2)$. It is close to 1 for nearby points and falls toward 0 for far points. The bandwidth knob $\gamma$ decides how local "similar" means: large $\gamma$ makes narrow neighborhoods; small $\gamma$ makes broad neighborhoods.

In [ ]:
x0_w = np.array([0.0])  # reference point.
xs_w = np.array([-3.0, -1.0, 0.0, 1.0, 3.0])[:, None]  # candidates at several distances.
gamma_w = 0.5  # moderate locality.
d2_w = np.sum((xs_w - x0_w) ** 2, axis=1)  # squared distances to the reference.
rbf_w = np.exp(-gamma_w * d2_w)  # RBF similarities.
print("squared distances:", d2_w)
print("RBF similarities:", np.round(rbf_w, 3))
assert round(float(rbf_w[2]), 3) == 1.0

▶ What you'll see: the reference has similarity 1 to itself and rapidly smaller values for distant points.

In [ ]:
grid3_w = np.linspace(-4, 4, 200)[:, None]
for gamma_plot_w in [0.1, 0.5, 2.0]:
    curve_w = np.exp(-gamma_plot_w * np.sum((grid3_w - x0_w) ** 2, axis=1))
    plt.plot(grid3_w.ravel(), curve_w, label=f"gamma={gamma_plot_w}")
plt.title("3: RBF bandwidth controls locality")
plt.xlabel("x")
plt.ylabel("K(x, 0)")
plt.legend()
plt.show()

▶ What you'll see: larger gamma produces a narrower peak, so only very close points receive substantial weight.

*Why it's done this way:* the exponential turns distance into a smooth positive similarity. Squared distance makes the penalty radial and differentiable, while $\gamma$ controls the bias-variance tradeoff: broad kernels are stable but coarse; narrow kernels are flexible but easier to overfit.

### 4. A Gram matrix stores all training similarities

For training data $X_1,\dots,X_n$, a kernel method usually forms the Gram matrix $G_{ij}=K(X_i,X_j)$. The diagonal is self-similarity, off-diagonal entries show pairwise resemblance, and valid kernels produce symmetric positive-semidefinite Gram matrices.

In [ ]:
X_w = np.array([[-1.5], [-0.5], [0.5], [1.5]])  # four toy training inputs.
gamma4_w = 0.8  # RBF bandwidth for the Gram matrix.
D2_w = (X_w - X_w.T) ** 2  # all pairwise squared distances in 1-D.
G_w = np.exp(-gamma4_w * D2_w)  # RBF Gram matrix.
print("Gram matrix:\n", np.round(G_w, 3))
print("symmetric:", np.allclose(G_w, G_w.T))

▶ What you'll see: diagonal entries are 1 and nearby points have larger off-diagonal similarities.

In [ ]:
eigs_w = np.linalg.eigvalsh(G_w)  # eigenvalues diagnose positive semidefiniteness.
print("Gram eigenvalues:", np.round(eigs_w, 6))
assert np.min(eigs_w) > -1e-10
plt.figure(figsize=(4, 3))
plt.imshow(G_w, cmap="viridis", aspect="auto")
plt.colorbar(label="kernel similarity")
plt.title("4: RBF Gram matrix")
plt.xlabel("training example")
plt.ylabel("training example")
plt.show()

▶ What you'll see: a bright diagonal band because adjacent inputs are similar and distant inputs are not.

*Why it's done this way:* once $G$ is available, many algorithms can be written using only similarities among examples. Positive semidefiniteness matters because it means the Gram matrix could have come from real inner products in some feature space, so the optimization remains geometrically well behaved.

### 5. Kernel prediction is a weighted sum of similarities

A kernel model often predicts with $f(x)=\sum_i \alpha_i K(x_i,x)$. The learned coefficients $\alpha_i$ say which training examples pull the prediction up or down; the kernel says how strongly the new point listens to each example.

In [ ]:
X5_w = np.array([[-2.0], [-0.5], [0.5], [2.0]])  # support/training locations.
y5_w = np.array([-1.0, -0.4, 0.6, 1.0])  # target signs or scores.
alpha5_w = np.array([-0.8, -0.3, 0.4, 0.7])  # hand-chosen coefficients for inspection.
x_new_w = np.array([[0.25]])  # point to score.
K_new_w = np.exp(-0.7 * (X5_w - x_new_w.T) ** 2).ravel()  # similarities to training points.
print("similarities to x=0.25:", np.round(K_new_w, 3))
print("coefficients:", alpha5_w)

▶ What you'll see: the new point listens most to nearby training points around -0.5 and 0.5.

In [ ]:
contrib5_w = alpha5_w * K_new_w  # each support point's contribution.
score5_w = float(np.sum(contrib5_w))  # kernel prediction.
print("contributions:", np.round(contrib5_w, 3))
print("kernel score:", round(score5_w, 3))
assert round(score5_w, 3) == 0.239

▶ What you'll see: negative and positive neighbors partially cancel, leaving a moderate positive score.

In [ ]:
grid5_w = np.linspace(-3, 3, 200)[:, None]
K_grid5_w = np.exp(-0.7 * (X5_w - grid5_w.T) ** 2)  # rows=train, columns=grid.
scores5_w = alpha5_w @ K_grid5_w  # f(x) over the grid.
plt.figure(figsize=(4.6, 3))
plt.plot(grid5_w.ravel(), scores5_w, color="navy")
plt.scatter(X5_w.ravel(), y5_w, color="orange", zorder=3)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("5: kernel score as weighted similarities")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.show()

▶ What you'll see: a smooth nonlinear score curve built from bumps centered at training examples.

*Why it's done this way:* the model never asks for coordinates of $\phi(x)$. It only asks, "How similar is this new point to each stored example?" The coefficients then combine that similarity evidence into a prediction, which is why kernel methods can be powerful but also memory-heavy.

### 6. Empirical risk, cost, and stability decide which kernel setting wins

The source lesson emphasizes that the raw training number is not the whole decision. For this toy kernel setting, the empirical losses are 0.213, 0.135, and 0.471; the method cost is 0.050; a tempting alternative scores 0.359; and a stabilizing knob reduces the baseline score by 20%.

In [ ]:
losses6_w = np.array([0.213, 0.135, 0.471])  # verified per-example losses.
R_S6_w = float(np.mean(losses6_w))  # empirical risk.
cost6_w = 0.050  # complexity or operational cost.
score6_w = R_S6_w + cost6_w  # full decision score.
print("empirical risk:", round(R_S6_w, 3))
print("risk + cost score:", round(score6_w, 3))
assert round(R_S6_w, 3) == 0.273
assert round(score6_w, 3) == 0.323

▶ What you'll see: the raw average is 0.273, but the selection score is 0.323 after cost.

In [ ]:
alternative6_w = 0.359  # more flexible alternative's decision score.
gap6_w = alternative6_w - score6_w  # absolute evidence gap.
relative_gap6_w = gap6_w / alternative6_w  # scale-aware gap.
stable6_w = 0.80 * score6_w  # 20% stabilized score.
choices6_w = np.array([score6_w, alternative6_w, stable6_w])
print("gap:", round(gap6_w, 3), "relative gap:", round(relative_gap6_w, 3))
print("stable score:", round(stable6_w, 3), "best:", round(float(np.min(choices6_w)), 3))
assert round(gap6_w, 3) == 0.036
assert round(relative_gap6_w, 3) == 0.100
assert round(stable6_w, 3) == 0.258

▶ What you'll see: the stabilized score is lowest, but the gap calculation also tells how strong the evidence is.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.bar(["baseline", "flexible", "stabilized"], choices6_w, color=["gray", "crimson", "teal"])
plt.title("6: choose by full score, not raw fit")
plt.ylabel("decision score (lower is better)")
plt.show()

▶ What you'll see: the stabilized option has the lowest full decision score in this verified toy case.

*Why it's done this way:* flexible kernels can make training loss look attractive, so model selection must include the cost or regularization implied by the method and the stability of the validation comparison. The correct unit of judgment is the full score, not the prettiest training fragment.

## 🛠️ Setup

In [ ]:
import numpy as np  # Import NumPy for arrays, kernels, distances, linear algebra, and assertions.
import matplotlib.pyplot as plt  # Import Matplotlib for heatmaps, curves, bars, and scatter plots.
np.random.seed(0)  # Fix the global random seed so every stochastic example is repeatable.

def linear_kernel(x, z):  # Compute the ordinary dot-product kernel.
    x = np.asarray(x, dtype=float)  # Convert the first vector to floats.
    z = np.asarray(z, dtype=float)  # Convert the second vector to floats.
    return float(x @ z)  # Return x^T z.

def polynomial_kernel(x, z, degree=2, c=1.0):  # Compute (x^T z + c)^degree.
    x = np.asarray(x, dtype=float)  # Convert the first vector to floats.
    z = np.asarray(z, dtype=float)  # Convert the second vector to floats.
    return float((x @ z + c) ** degree)  # Return the polynomial similarity.

def rbf_kernel_matrix(A, B, gamma=1.0):  # Compute all RBF similarities between rows of A and rows of B.
    A = np.asarray(A, dtype=float)  # Ensure A is a numeric matrix.
    B = np.asarray(B, dtype=float)  # Ensure B is a numeric matrix.
    d2 = np.sum((A[:, None, :] - B[None, :, :]) ** 2, axis=2)  # Pairwise squared distances.
    return np.exp(-gamma * d2)  # Convert distances into similarities.

def show_kernel_matrix(M, title):  # Display a compact heatmap for a Gram matrix.
    plt.figure(figsize=(4, 3))  # Start a small figure.
    plt.imshow(M, cmap="viridis", aspect="auto")  # Draw matrix values as colors.
    plt.colorbar(label="kernel value")  # Add a readable color scale.
    plt.title(title)  # Label the plot.
    plt.xlabel("example")  # Label columns.
    plt.ylabel("example")  # Label rows.
    plt.show()  # Display the figure.

## 🟢 Basics (warm-up)

### Basic 1 — Compute an ordinary dot-product kernel

**Goal.** Start with the simplest kernel, because every kernel method generalizes the idea of comparing examples by an inner product. We build it in 2 steps.

In [ ]:
x_b1 = np.array([2.0, 1.0])  # Define one input vector.
z_b1 = np.array([1.0, 3.0])  # Define a second input vector.
products_b1 = x_b1 * z_b1  # Multiply coordinate by coordinate before summing.
print("coordinate products:", products_b1)  # Inspect each contribution to x dot z.

▶ What you'll see: each coordinate contributes separately to the final similarity.

In [ ]:
k_b1 = linear_kernel(x_b1, z_b1)  # Compute the ordinary dot product.
print("linear kernel:", round(k_b1, 3))  # Inspect K(x,z).
assert round(k_b1, 3) == 5.0  # Verify 2*1 + 1*3 = 5.
plt.figure(figsize=(4, 3))  # Create a small contribution plot.
plt.bar(["coord 0", "coord 1"], products_b1, color="teal")  # Show dot-product pieces.
plt.title("Basic 1: dot-product contributions")  # Title the plot.
plt.ylabel("x_j z_j")  # Label the contribution scale.
plt.show()  # Display the bar chart.

▶ What you'll see: the two bars sum to the printed kernel value 5.

👀 Takeaway: the linear kernel is just ordinary feature alignment.

### Basic 2 — Build an explicit quadratic feature map

**Goal.** Expand a 2-D vector into quadratic features, because the kernel trick replaces this explicit expansion with a cheaper formula. We build it in 2 steps.

In [ ]:
x_b2 = np.array([2.0, 1.0])  # Define the original 2-D input.
phi_b2 = np.array([x_b2[0]**2, np.sqrt(2)*x_b2[0]*x_b2[1], x_b2[1]**2])  # Build degree-2 features.
print("x:", x_b2)  # Inspect the original coordinates.
print("phi(x):", np.round(phi_b2, 3))  # Inspect the expanded coordinates.

▶ What you'll see: a 2-D input becomes a 3-D vector containing square and interaction terms.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a compact feature bar chart.
plt.bar(["x0²", "√2x0x1", "x1²"], phi_b2, color="orange")  # Visualize the expanded coordinates.
plt.title("Basic 2: explicit quadratic map")  # Title the plot.
plt.ylabel("feature value")  # Label the scale.
plt.show()  # Display the feature map.

▶ What you'll see: the cross term is scaled by √2 so the polynomial identity works exactly.

👀 Takeaway: nonlinear feature maps create interactions that a linear model can use.

### Basic 3 — Verify the degree-2 polynomial identity

**Goal.** Check that $\phi(x)^\top\phi(z)=(x^\top z)^2$, because this equality is the kernel trick in its smallest concrete form. We build it in 2 steps.

In [ ]:
x_b3 = np.array([2.0, 1.0])  # Define the first vector.
z_b3 = np.array([1.0, 3.0])  # Define the second vector.
phi_x_b3 = np.array([x_b3[0]**2, np.sqrt(2)*x_b3[0]*x_b3[1], x_b3[1]**2])  # Explicit phi(x).
phi_z_b3 = np.array([z_b3[0]**2, np.sqrt(2)*z_b3[0]*z_b3[1], z_b3[1]**2])  # Explicit phi(z).
print("phi_x:", np.round(phi_x_b3, 3), "phi_z:", np.round(phi_z_b3, 3))  # Inspect expanded vectors.

▶ What you'll see: both vectors are expanded before taking an ordinary dot product.

In [ ]:
explicit_b3 = float(phi_x_b3 @ phi_z_b3)  # Compute the expanded-space inner product.
implicit_b3 = float((x_b3 @ z_b3) ** 2)  # Compute the kernel directly in input space.
print("explicit:", round(explicit_b3, 3), "implicit:", round(implicit_b3, 3))  # Compare both routes.
assert round(explicit_b3, 3) == 25.0  # Verify the expanded calculation.
assert round(implicit_b3, 3) == 25.0  # Verify the kernel calculation.
plt.figure(figsize=(4, 3))  # Create a comparison plot.
plt.bar(["explicit φ dot", "implicit K"], [explicit_b3, implicit_b3], color=["teal", "purple"])  # Compare values.
plt.title("Basic 3: same inner product")  # Title the plot.
plt.show()  # Display the bars.

▶ What you'll see: the two bars are identical.

👀 Takeaway: a valid kernel can compute an expanded-space dot product without materializing the expanded vectors.

### Basic 4 — Add an offset to a polynomial kernel

**Goal.** Use $K(x,z)=(x^\top z+c)^d$, because the offset keeps lower-degree evidence along with high-degree interactions. We build it in 2 steps.

In [ ]:
x_b4 = np.array([1.0, 2.0])  # Define the first vector.
z_b4 = np.array([2.0, 1.0])  # Define the second vector.
dot_b4 = linear_kernel(x_b4, z_b4)  # Compute the raw dot product.
print("dot:", dot_b4)  # Inspect the base similarity before offset and power.

▶ What you'll see: the raw dot product is the ingredient the polynomial kernel transforms.

In [ ]:
poly_no_offset_b4 = polynomial_kernel(x_b4, z_b4, degree=2, c=0.0)  # Compute (x dot z)^2.
poly_offset_b4 = polynomial_kernel(x_b4, z_b4, degree=2, c=1.0)  # Compute (x dot z + 1)^2.
print("without offset:", round(poly_no_offset_b4, 3), "with offset:", round(poly_offset_b4, 3))  # Compare kernels.
assert round(poly_no_offset_b4, 3) == 16.0  # Verify 4^2.
assert round(poly_offset_b4, 3) == 25.0  # Verify (4+1)^2.
plt.figure(figsize=(4, 3))  # Create a small comparison chart.
plt.bar(["c=0", "c=1"], [poly_no_offset_b4, poly_offset_b4], color=["gray", "teal"])  # Show offset effect.
plt.title("Basic 4: polynomial offset")  # Title the plot.
plt.ylabel("K(x,z)")  # Label kernel value.
plt.show()  # Display the chart.

▶ What you'll see: adding the offset increases similarity because it adds lower-order terms.

👀 Takeaway: the offset controls whether the polynomial feature space includes simpler terms.

### Basic 5 — Compute one RBF similarity

**Goal.** Convert distance into similarity, because RBF kernels judge examples by closeness rather than coordinate alignment. We build it in 2 steps.

In [ ]:
x_b5 = np.array([0.0, 0.0])  # Define a reference point.
z_b5 = np.array([1.0, 2.0])  # Define a second point.
gamma_b5 = 0.5  # Choose an RBF bandwidth parameter.
d2_b5 = float(np.sum((x_b5 - z_b5) ** 2))  # Compute squared Euclidean distance.
print("squared distance:", round(d2_b5, 3))  # Inspect the distance penalty.

▶ What you'll see: the squared distance is 5.

In [ ]:
k_b5 = float(np.exp(-gamma_b5 * d2_b5))  # Compute the RBF kernel value.
print("RBF similarity:", round(k_b5, 3))  # Inspect the closeness score.
assert round(k_b5, 3) == 0.082  # Verify exp(-2.5).
plt.figure(figsize=(4, 3))  # Create a point plot.
plt.scatter([x_b5[0], z_b5[0]], [x_b5[1], z_b5[1]], s=100, color=["teal", "orange"])  # Draw the two points.
plt.plot([x_b5[0], z_b5[0]], [x_b5[1], z_b5[1]], color="gray", linestyle="--")  # Show the distance.
plt.title("Basic 5: distance behind RBF")  # Title the plot.
plt.axis("equal")  # Keep geometry honest.
plt.show()  # Display the plot.

▶ What you'll see: the two points are separated, so their RBF similarity is small.

👀 Takeaway: RBF kernels turn near points into high similarity and far points into low similarity.

### Basic 6 — See gamma change locality

**Goal.** Compare RBF curves for several $\gamma$ values, because bandwidth controls how flexible the kernel method can be. We build it in 2 steps.

In [ ]:
grid_b6 = np.linspace(-3, 3, 121)[:, None]  # Build a 1-D grid of candidate points.
center_b6 = np.array([[0.0]])  # Use zero as the reference point.
gammas_b6 = np.array([0.1, 0.5, 2.0])  # Choose broad, medium, and narrow kernels.
print("gammas:", gammas_b6)  # Inspect the bandwidth settings.

▶ What you'll see: the example compares three locality settings.

In [ ]:
for gamma_b6 in gammas_b6:  # Plot one RBF curve per gamma.
    sims_b6 = rbf_kernel_matrix(grid_b6, center_b6, gamma=float(gamma_b6)).ravel()  # Similarity to zero.
    plt.plot(grid_b6.ravel(), sims_b6, label=f"γ={gamma_b6}")  # Draw the curve.
plt.title("Basic 6: gamma controls RBF width")  # Title the plot.
plt.xlabel("x")  # Label input axis.
plt.ylabel("K(x,0)")  # Label similarity axis.
plt.legend()  # Show gamma labels.
plt.show()  # Display the curves.

▶ What you'll see: larger gamma narrows the similarity bump around zero.

👀 Takeaway: gamma is a bias-variance knob for RBF kernel methods.

### Basic 7 — Build a tiny Gram matrix

**Goal.** Store all pairwise kernel values, because kernel methods train from Gram matrices rather than explicit features. We build it in 2 steps.

In [ ]:
X_b7 = np.array([[-1.0], [0.0], [1.0]])  # Define three training examples.
G_b7 = rbf_kernel_matrix(X_b7, X_b7, gamma=1.0)  # Compute all pairwise RBF similarities.
print("G_b7:\n", np.round(G_b7, 3))  # Inspect the Gram matrix.

▶ What you'll see: self-similarities are 1 and neighbors one unit apart have similarity about 0.368.

In [ ]:
assert np.allclose(np.diag(G_b7), 1.0)  # Verify every example has kernel value 1 with itself.
assert round(float(G_b7[0, 1]), 3) == 0.368  # Verify exp(-1).
show_kernel_matrix(G_b7, "Basic 7: RBF Gram matrix")  # Visualize the matrix.

▶ What you'll see: the diagonal is brightest and off-diagonal values fade with distance.

👀 Takeaway: the Gram matrix is the training set viewed entirely through similarities.

### Basic 8 — Check Gram matrix symmetry

**Goal.** Verify $K(x,z)=K(z,x)$, because ordinary inner products are symmetric and valid kernel Gram matrices inherit that property. We build it in 2 steps.

In [ ]:
X_b8 = np.array([[-2.0], [-0.5], [1.0]])  # Define three inputs.
G_b8 = rbf_kernel_matrix(X_b8, X_b8, gamma=0.7)  # Compute a symmetric kernel matrix.
symmetry_error_b8 = np.max(np.abs(G_b8 - G_b8.T))  # Measure the largest asymmetry.
print("max symmetry error:", symmetry_error_b8)  # Inspect numerical symmetry.

▶ What you'll see: the symmetry error is zero up to floating-point precision.

In [ ]:
assert symmetry_error_b8 < 1e-12  # Verify symmetry numerically.
plt.figure(figsize=(4, 3))  # Create a comparison heatmap.
plt.imshow(G_b8 - G_b8.T, cmap="coolwarm", aspect="auto")  # Show any asymmetry.
plt.colorbar(label="G - Gᵀ")  # Add a scale.
plt.title("Basic 8: symmetry check")  # Title the plot.
plt.show()  # Display the heatmap.

▶ What you'll see: the heatmap is essentially zero everywhere.

👀 Takeaway: symmetry is a quick sanity check for a kernel implementation.

### Basic 9 — Compute a kernel prediction from coefficients

**Goal.** Evaluate $f(x)=\sum_i\alpha_iK(x_i,x)$, because kernel models predict by weighting similarities to stored examples. We build it in 2 steps.

In [ ]:
X_b9 = np.array([[-1.0], [0.0], [1.0]])  # Training locations.
alpha_b9 = np.array([-1.0, 0.2, 1.0])  # Learned or hand-set coefficients.
x_new_b9 = np.array([[0.5]])  # New point to score.
k_new_b9 = rbf_kernel_matrix(X_b9, x_new_b9, gamma=1.0).ravel()  # Similarities to the new point.
print("similarities:", np.round(k_new_b9, 3))  # Inspect how much each stored example matters.

▶ What you'll see: the new point is more similar to 0 and 1 than to -1.

In [ ]:
contrib_b9 = alpha_b9 * k_new_b9  # Compute each stored example's contribution.
score_b9 = float(np.sum(contrib_b9))  # Sum contributions into a prediction score.
print("contributions:", np.round(contrib_b9, 3), "score:", round(score_b9, 3))  # Inspect prediction arithmetic.
assert round(score_b9, 3) == 0.829  # Verify the worked score.
plt.figure(figsize=(4, 3))  # Create a contribution plot.
plt.bar(["x=-1", "x=0", "x=1"], contrib_b9, color="purple")  # Show each term in the sum.
plt.axhline(0, color="black", linewidth=0.8)  # Add zero reference.
plt.title("Basic 9: weighted kernel contributions")  # Title the plot.
plt.show()  # Display the chart.

▶ What you'll see: the positive nearby example dominates the final score.

👀 Takeaway: kernel predictions are similarity-weighted sums over training examples.

### Basic 10 — Average three empirical losses

**Goal.** Recompute the lesson's empirical risk, because model selection begins with the average loss over examples. We build it in 2 steps.

In [ ]:
losses_b10 = np.array([0.213, 0.135, 0.471])  # Use the verified per-example losses from the lesson.
sum_b10 = float(np.sum(losses_b10))  # Sum the losses before averaging.
print("loss sum:", round(sum_b10, 3))  # Inspect the numerator of empirical risk.

▶ What you'll see: the three losses sum to 0.819.

In [ ]:
risk_b10 = float(np.mean(losses_b10))  # Average the losses.
print("empirical risk:", round(risk_b10, 3))  # Inspect the training score.
assert round(sum_b10, 3) == 0.819  # Verify the lesson sum.
assert round(risk_b10, 3) == 0.273  # Verify the lesson average.
plt.figure(figsize=(4, 3))  # Create a loss bar chart.
plt.bar(["ex1", "ex2", "ex3"], losses_b10, color="teal")  # Show individual losses.
plt.axhline(risk_b10, color="red", linestyle="--", label="average")  # Mark the empirical risk.
plt.title("Basic 10: empirical risk is an average")  # Title the plot.
plt.legend()  # Show the average label.
plt.show()  # Display the chart.

▶ What you'll see: the average line summarizes the per-example losses.

👀 Takeaway: empirical risk is the average training loss, not a single hand-picked example.

## 🟡 Easy

### Easy 1 — Compare linear and polynomial Gram matrices

**Goal.** Build two Gram matrices from the same data, because changing the kernel changes what the learner considers similar. We build it in 3 steps.

In [ ]:
X_e1 = np.array([[-1.0, 0.5], [0.0, 1.0], [1.0, 0.5], [2.0, 1.0]])  # Four small 2-D examples.
G_linear_e1 = X_e1 @ X_e1.T  # Linear Gram matrix from ordinary dot products.
print("linear Gram:\n", np.round(G_linear_e1, 3))  # Inspect linear similarities.

▶ What you'll see: the linear matrix can be negative when points point in opposite directions.

In [ ]:
G_poly_e1 = (X_e1 @ X_e1.T + 1.0) ** 2  # Degree-2 polynomial Gram matrix with offset.
print("polynomial Gram:\n", np.round(G_poly_e1, 3))  # Inspect nonlinear similarities.
assert round(float(G_poly_e1[0, 2]), 3) == 0.062  # Verify (-0.75+1)^2.

In [ ]:
fig, ax_e1 = plt.subplots(1, 2, figsize=(7, 3))  # Create side-by-side heatmaps.
ax_e1[0].imshow(G_linear_e1, cmap="viridis", aspect="auto")  # Draw the linear Gram matrix.
ax_e1[0].set_title("linear")  # Title first heatmap.
ax_e1[1].imshow(G_poly_e1, cmap="viridis", aspect="auto")  # Draw the polynomial Gram matrix.
ax_e1[1].set_title("polynomial")  # Title second heatmap.
plt.suptitle("Easy 1: same data, different similarities")  # Overall title.
plt.show()  # Display both matrices.

▶ What you'll see: the polynomial kernel changes the similarity pattern by adding nonlinear interactions.

👀 Takeaway: choosing a kernel is choosing the geometry used by the model.

### Easy 2 — Use RBF kernel regression by hand

**Goal.** Predict a smooth target by kernel-weighted averaging, because local similarities can act as nonparametric regression weights. We build it in 4 steps.

In [ ]:
X_e2 = np.array([[-2.0], [-1.0], [0.0], [1.0], [2.0]])  # Training inputs.
y_e2 = np.array([-0.9, -0.8, 0.0, 0.8, 0.9])  # Smooth target values.
x_new_e2 = np.array([[0.25]])  # Query point.
gamma_e2 = 1.0  # Locality setting.
print("query:", x_new_e2.ravel()[0])  # Inspect the point to predict.

▶ What you'll see: the query lies between 0 and 1.

In [ ]:
weights_e2 = rbf_kernel_matrix(X_e2, x_new_e2, gamma=gamma_e2).ravel()  # Similarities from each training point to the query.
weights_e2 = weights_e2 / np.sum(weights_e2)  # Normalize similarities into averaging weights.
print("normalized weights:", np.round(weights_e2, 3))  # Inspect local attention.
assert round(float(np.sum(weights_e2)), 3) == 1.0  # Verify weights form an average.

In [ ]:
pred_e2 = float(np.sum(weights_e2 * y_e2))  # Weighted average prediction.
print("kernel regression prediction:", round(pred_e2, 3))  # Inspect the predicted value.
assert round(pred_e2, 3) == 0.183  # Verify the worked result.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a regression plot.
plt.scatter(X_e2.ravel(), y_e2, color="orange", label="training")  # Plot training targets.
plt.scatter([x_new_e2.ravel()[0]], [pred_e2], color="teal", s=90, label="prediction")  # Plot query prediction.
for xi_e2, wi_e2 in zip(X_e2.ravel(), weights_e2):  # Draw attention weights as vertical markers.
    plt.plot([xi_e2, xi_e2], [-1.1, -1.1 + wi_e2], color="gray", linewidth=4)  # Show relative weight.
plt.title("Easy 2: RBF-weighted average")  # Title the plot.
plt.legend()  # Show labels.
plt.show()  # Display the figure.

▶ What you'll see: points near the query carry the largest weights and pull the prediction positive.

👀 Takeaway: RBF regression predicts locally by normalizing kernel similarities into weights.

### Easy 3 — Add cost to the raw empirical risk

**Goal.** Compute the full selection score, because the lesson's decision uses empirical risk plus a method cost. We build it in 3 steps.

In [ ]:
losses_e3 = np.array([0.213, 0.135, 0.471])  # Verified toy losses.
cost_e3 = 0.050  # Complexity, regularization, or operational cost.
risk_e3 = float(np.mean(losses_e3))  # Average loss.
print("risk:", round(risk_e3, 3), "cost:", round(cost_e3, 3))  # Inspect both terms.

▶ What you'll see: the raw training average is only one part of the score.

In [ ]:
score_e3 = risk_e3 + cost_e3  # Add the cost term.
print("decision score:", round(score_e3, 3))  # Inspect the full selection quantity.
assert round(score_e3, 3) == 0.323  # Verify the lesson score.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a score breakdown chart.
plt.bar(["risk", "cost", "score"], [risk_e3, cost_e3, score_e3], color=["teal", "orange", "purple"])  # Show additive pieces.
plt.title("Easy 3: risk plus cost")  # Title the plot.
plt.ylabel("value")  # Label the scale.
plt.show()  # Display the chart.

▶ What you'll see: the score bar is higher than raw risk because it includes the guardrail cost.

👀 Takeaway: model selection should optimize the full score implied by the method, not just raw fit.

### Easy 4 — Compute a validation gap and relative gap

**Goal.** Compare a baseline to a tempting flexible alternative, because tiny score wins can disappear under resampling noise. We build it in 3 steps.

In [ ]:
baseline_e4 = 0.323  # Baseline kernel setting score.
flexible_e4 = 0.359  # More flexible alternative score.
gap_e4 = flexible_e4 - baseline_e4  # Absolute score difference.
print("baseline:", baseline_e4, "flexible:", flexible_e4)  # Inspect compared scores.

▶ What you'll see: lower is better, so the baseline is ahead before considering uncertainty.

In [ ]:
relative_gap_e4 = gap_e4 / flexible_e4  # Express the gap on the alternative's scale.
print("gap:", round(gap_e4, 3), "relative gap:", round(relative_gap_e4, 3))  # Inspect evidence strength.
assert round(gap_e4, 3) == 0.036  # Verify the lesson gap.
assert round(relative_gap_e4, 3) == 0.100  # Verify the lesson relative gap.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a comparison bar chart.
plt.bar(["baseline", "flexible"], [baseline_e4, flexible_e4], color=["teal", "crimson"])  # Compare decision scores.
plt.title("Easy 4: validation gap")  # Title the plot.
plt.ylabel("score (lower is better)")  # Label the score scale.
plt.show()  # Display the chart.

▶ What you'll see: the flexible method is worse by 0.036, about 10% of its score.

👀 Takeaway: the gap is the evidence for preferring one kernel setting over another.

### Easy 5 — Pick the stabilized kernel setting

**Goal.** Apply a 20% stabilization improvement and choose the lowest full score, because regularization can make a flexible method more durable. We build it in 3 steps.

In [ ]:
baseline_e5 = 0.323  # Baseline full score.
flexible_e5 = 0.359  # Flexible alternative score.
stabilized_e5 = 0.80 * baseline_e5  # Stabilizing knob reduces the baseline score by 20%.
print("stabilized score:", round(stabilized_e5, 3))  # Inspect the improved score.

▶ What you'll see: stabilization lowers the score to about 0.258.

In [ ]:
scores_e5 = np.array([baseline_e5, flexible_e5, stabilized_e5])  # Collect all candidate scores.
labels_e5 = np.array(["baseline", "flexible", "stabilized"])  # Candidate names.
best_idx_e5 = int(np.argmin(scores_e5))  # Choose the lowest score.
print("best setting:", labels_e5[best_idx_e5], "score:", round(float(scores_e5[best_idx_e5]), 3))  # Inspect the decision.
assert round(float(stabilized_e5), 3) == 0.258  # Verify the lesson stabilized value.
assert labels_e5[best_idx_e5] == "stabilized"  # Verify the final winner.

In [ ]:
plt.figure(figsize=(4.8, 3))  # Create a model-selection plot.
plt.bar(labels_e5, scores_e5, color=["gray", "crimson", "teal"])  # Show all candidate scores.
plt.title("Easy 5: final score comparison")  # Title the plot.
plt.ylabel("score (lower is better)")  # Label score scale.
plt.show()  # Display the chart.

▶ What you'll see: the stabilized score is the smallest bar.

👀 Takeaway: regularization or stabilization should be judged by the final decision score.

## 🔴 Advanced

### Advanced 1 — Solve kernel ridge regression in the dual

**Goal.** Fit coefficients $\alpha=(K+\lambda I)^{-1}y$, because kernel ridge regression learns in terms of Gram matrices rather than explicit feature weights. We build it in 4 steps.

In [ ]:
X_a1 = np.linspace(-2, 2, 7)[:, None]  # Training inputs.
y_a1 = np.sin(X_a1.ravel())  # Smooth nonlinear targets.
gamma_a1 = 0.8  # RBF bandwidth.
lam_a1 = 0.1  # Ridge regularization.
K_a1 = rbf_kernel_matrix(X_a1, X_a1, gamma=gamma_a1)  # Training Gram matrix.
print("K shape:", K_a1.shape, "lambda:", lam_a1)  # Inspect system size and regularization.

▶ What you'll see: the dual system has one coefficient per training example.

In [ ]:
alpha_a1 = np.linalg.solve(K_a1 + lam_a1 * np.eye(len(X_a1)), y_a1)  # Solve the regularized dual linear system.
print("alpha:", np.round(alpha_a1, 3))  # Inspect learned example weights.
assert alpha_a1.shape == (7,)  # Verify one coefficient per example.

In [ ]:
grid_a1 = np.linspace(-2.5, 2.5, 120)[:, None]  # Prediction grid.
K_grid_a1 = rbf_kernel_matrix(grid_a1, X_a1, gamma=gamma_a1)  # Similarities from grid points to training points.
pred_a1 = K_grid_a1 @ alpha_a1  # Dual prediction f(x)=k(x,X) alpha.
train_pred_a1 = K_a1 @ alpha_a1  # Predictions at training inputs.
rmse_a1 = float(np.sqrt(np.mean((train_pred_a1 - y_a1) ** 2)))  # Training RMSE.
print("train RMSE:", round(rmse_a1, 3))  # Inspect fit quality.
assert rmse_a1 < 0.12  # Verify the small toy fit is reasonable.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a regression curve plot.
plt.plot(grid_a1.ravel(), pred_a1, color="teal", label="kernel ridge")  # Plot model predictions.
plt.scatter(X_a1.ravel(), y_a1, color="orange", label="training")  # Plot data.
plt.title("Advanced 1: dual kernel ridge regression")  # Title the plot.
plt.legend()  # Show labels.
plt.show()  # Display the curve.

▶ What you'll see: the curve smoothly follows the sine-like training targets.

👀 Takeaway: kernel ridge learns coefficients over examples, with λ stabilizing the Gram-system solve.

### Advanced 2 — Sweep RBF bandwidth and watch validation error

**Goal.** Tune $\gamma$ with validation data, because bandwidth controls underfitting versus overfitting. We build it in 4 steps.

In [ ]:
X_train_a2 = np.linspace(-3, 3, 13)[:, None]  # Training inputs.
y_train_a2 = np.sin(X_train_a2.ravel())  # Training targets.
X_val_a2 = np.linspace(-2.7, 2.7, 10)[:, None]  # Validation inputs between training points.
y_val_a2 = np.sin(X_val_a2.ravel())  # Validation targets.
gammas_a2 = np.array([0.05, 0.2, 0.8, 3.0])  # Candidate bandwidths.
print("gamma grid:", gammas_a2)  # Inspect candidates.

▶ What you'll see: the sweep compares broad through narrow RBF kernels.

In [ ]:
val_rmse_a2 = []  # Store validation errors.
train_rmse_a2 = []  # Store training errors.
for gamma_a2 in gammas_a2:  # Fit one kernel ridge model per gamma.
    K_train_a2 = rbf_kernel_matrix(X_train_a2, X_train_a2, gamma=float(gamma_a2))  # Training Gram matrix.
    alpha_a2 = np.linalg.solve(K_train_a2 + 0.05 * np.eye(len(X_train_a2)), y_train_a2)  # Regularized dual coefficients.
    pred_train_a2 = K_train_a2 @ alpha_a2  # Training predictions.
    pred_val_a2 = rbf_kernel_matrix(X_val_a2, X_train_a2, gamma=float(gamma_a2)) @ alpha_a2  # Validation predictions.
    train_rmse_a2.append(float(np.sqrt(np.mean((pred_train_a2 - y_train_a2) ** 2))))  # Store train RMSE.
    val_rmse_a2.append(float(np.sqrt(np.mean((pred_val_a2 - y_val_a2) ** 2))))  # Store validation RMSE.
print("train RMSE:", np.round(train_rmse_a2, 3))  # Inspect fit.
print("validation RMSE:", np.round(val_rmse_a2, 3))  # Inspect generalization.

In [ ]:
best_idx_a2 = int(np.argmin(val_rmse_a2))  # Select gamma by validation error.
best_gamma_a2 = float(gammas_a2[best_idx_a2])  # Read best gamma.
print("best gamma:", best_gamma_a2)  # Inspect selected bandwidth.
assert best_gamma_a2 in gammas_a2  # Verify selection came from the grid.

In [ ]:
plt.figure(figsize=(5, 3))  # Create bandwidth sweep plot.
plt.plot(gammas_a2, train_rmse_a2, marker="o", label="train")  # Plot train errors.
plt.plot(gammas_a2, val_rmse_a2, marker="s", label="validation")  # Plot validation errors.
plt.xscale("log")  # Log scale makes the gamma grid readable.
plt.axvline(best_gamma_a2, color="red", linestyle="--", label="best γ")  # Mark selected gamma.
plt.title("Advanced 2: bandwidth validation sweep")  # Title the plot.
plt.xlabel("gamma")  # Label bandwidth axis.
plt.ylabel("RMSE")  # Label error axis.
plt.legend()  # Show labels.
plt.show()  # Display the sweep.

▶ What you'll see: validation, not training alone, identifies the most useful bandwidth.

👀 Takeaway: kernel flexibility should be tuned on held-out data.

### Advanced 3 — Compare regularization strengths

**Goal.** Sweep λ in kernel ridge regression, because regularization controls how much the model trusts flexible kernel fits. We build it in 4 steps.

In [ ]:
X_a3 = np.linspace(-2.5, 2.5, 11)[:, None]  # Training inputs.
y_a3 = np.sin(X_a3.ravel()) + 0.05 * np.cos(3 * X_a3.ravel())  # Slightly wiggly targets.
lams_a3 = np.array([0.001, 0.01, 0.1, 1.0])  # Candidate ridge strengths.
gamma_a3 = 0.9  # Fixed RBF bandwidth.
print("lambda grid:", lams_a3)  # Inspect regularization settings.

▶ What you'll see: the sweep ranges from weak to strong shrinkage.

In [ ]:
norms_a3 = []  # Store coefficient norms.
errors_a3 = []  # Store training errors.
K_a3 = rbf_kernel_matrix(X_a3, X_a3, gamma=gamma_a3)  # Fixed Gram matrix.
for lam_loop_a3 in lams_a3:  # Fit one model per lambda.
    alpha_loop_a3 = np.linalg.solve(K_a3 + lam_loop_a3 * np.eye(len(X_a3)), y_a3)  # Regularized solve.
    pred_loop_a3 = K_a3 @ alpha_loop_a3  # Training predictions.
    norms_a3.append(float(np.linalg.norm(alpha_loop_a3)))  # Coefficient size.
    errors_a3.append(float(np.sqrt(np.mean((pred_loop_a3 - y_a3) ** 2))))  # Training RMSE.
print("alpha norms:", np.round(norms_a3, 3))  # Inspect shrinkage.
print("train RMSE:", np.round(errors_a3, 3))  # Inspect fit cost.

In [ ]:
assert norms_a3[0] > norms_a3[-1]  # Verify stronger regularization shrinks coefficients.
assert errors_a3[-1] > errors_a3[0]  # Verify strong regularization sacrifices training fit.
print("weak λ norm:", round(norms_a3[0], 3), "strong λ norm:", round(norms_a3[-1], 3))  # Inspect the tradeoff.

In [ ]:
fig, ax_a3 = plt.subplots(1, 2, figsize=(7, 3))  # Create two diagnostic panels.
ax_a3[0].plot(lams_a3, norms_a3, marker="o", color="teal")  # Plot coefficient norm.
ax_a3[0].set_xscale("log")  # Log lambda axis.
ax_a3[0].set_title("coefficient norm")  # Title first panel.
ax_a3[1].plot(lams_a3, errors_a3, marker="s", color="purple")  # Plot training error.
ax_a3[1].set_xscale("log")  # Log lambda axis.
ax_a3[1].set_title("training RMSE")  # Title second panel.
plt.suptitle("Advanced 3: regularization tradeoff")  # Overall title.
plt.show()  # Display panels.

▶ What you'll see: larger λ shrinks coefficients but increases training error.

👀 Takeaway: λ is the explicit cost term that prevents flexible kernels from chasing brittle variation.

### Advanced 4 — Approximate an RBF kernel with random Fourier features

**Goal.** Build a finite feature map whose dot products approximate an RBF kernel, because random features trade exact kernel memory for explicit vectors. We build it in 4 steps.

In [ ]:
rng_a4 = np.random.default_rng(4)  # Local reproducible generator.
x_a4 = np.array([[0.2]])  # First point.
z_a4 = np.array([[1.0]])  # Second point.
gamma_a4 = 0.7  # RBF bandwidth.
D_a4 = 400  # Number of random Fourier features.
W_a4 = rng_a4.normal(0.0, np.sqrt(2 * gamma_a4), size=(1, D_a4))  # Random frequencies for RBF approximation.
b_a4 = rng_a4.uniform(0.0, 2 * np.pi, size=D_a4)  # Random phases.
print("random feature count:", D_a4)  # Inspect approximation size.

▶ What you'll see: the approximation uses hundreds of explicit random coordinates.

In [ ]:
phi_x_a4 = np.sqrt(2.0 / D_a4) * np.cos(x_a4 @ W_a4 + b_a4)  # Random Fourier features for x.
phi_z_a4 = np.sqrt(2.0 / D_a4) * np.cos(z_a4 @ W_a4 + b_a4)  # Random Fourier features for z.
approx_a4 = float(phi_x_a4 @ phi_z_a4.T)  # Approximate kernel by a dot product.
exact_a4 = float(rbf_kernel_matrix(x_a4, z_a4, gamma=gamma_a4)[0, 0])  # Exact RBF kernel.
print("approx:", round(approx_a4, 3), "exact:", round(exact_a4, 3))  # Compare approximation.
assert abs(approx_a4 - exact_a4) < 0.12  # Verify the approximation is close for this seed.

In [ ]:
Ds_a4 = np.array([20, 50, 100, 400])  # Feature counts to compare.
errs_a4 = []  # Store approximation errors.
for D_loop_a4 in Ds_a4:  # Rebuild approximation at each size.
    rng_loop_a4 = np.random.default_rng(4)  # Reuse seed so feature count is the changed variable.
    W_loop_a4 = rng_loop_a4.normal(0.0, np.sqrt(2 * gamma_a4), size=(1, D_loop_a4))  # Frequencies.
    b_loop_a4 = rng_loop_a4.uniform(0.0, 2 * np.pi, size=D_loop_a4)  # Phases.
    px_loop_a4 = np.sqrt(2.0 / D_loop_a4) * np.cos(x_a4 @ W_loop_a4 + b_loop_a4)  # Features for x.
    pz_loop_a4 = np.sqrt(2.0 / D_loop_a4) * np.cos(z_a4 @ W_loop_a4 + b_loop_a4)  # Features for z.
    errs_a4.append(abs(float(px_loop_a4 @ pz_loop_a4.T) - exact_a4))  # Absolute error.
print("absolute errors:", np.round(errs_a4, 3))  # Inspect approximation quality.

In [ ]:
plt.figure(figsize=(5, 3))  # Create approximation-error plot.
plt.plot(Ds_a4, errs_a4, marker="o", color="teal")  # Plot error by feature count.
plt.title("Advanced 4: random-feature approximation")  # Title the plot.
plt.xlabel("number of random features")  # Label feature count.
plt.ylabel("|approx - exact|")  # Label error.
plt.show()  # Display the plot.

▶ What you'll see: more random features usually make the explicit dot product closer to the exact RBF kernel.

👀 Takeaway: random features approximate the kernel trick when exact Gram matrices are too large.

### Advanced 5 — Show why Gram matrices can be expensive

**Goal.** Estimate kernel storage and prediction cost as training size grows, because exact kernel methods remember similarities to many examples. We build it in 3 steps.

In [ ]:
sizes_a5 = np.array([100, 500, 1000, 5000])  # Training-set sizes to compare.
bytes_per_float_a5 = 8  # Float64 storage.
gram_mb_a5 = sizes_a5 ** 2 * bytes_per_float_a5 / 1_000_000  # Dense Gram matrix memory in MB.
print("Gram MB:", np.round(gram_mb_a5, 2))  # Inspect quadratic storage growth.
assert round(float(gram_mb_a5[2]), 2) == 8.0  # Verify 1000x1000 float64 matrix is 8 MB.

▶ What you'll see: memory grows quadratically with the number of training examples.

In [ ]:
support_counts_a5 = np.array([20, 100, 500, 2000])  # Number of stored support examples used at prediction time.
query_cost_a5 = support_counts_a5  # One kernel evaluation per support example for a single query.
print("kernel evaluations per query:", query_cost_a5)  # Inspect linear prediction cost in support count.

In [ ]:
fig, ax_a5 = plt.subplots(1, 2, figsize=(7, 3))  # Create two cost panels.
ax_a5[0].plot(sizes_a5, gram_mb_a5, marker="o", color="crimson")  # Plot memory cost.
ax_a5[0].set_title("Gram memory")  # Title first panel.
ax_a5[0].set_xlabel("n training examples")  # Label x-axis.
ax_a5[0].set_ylabel("MB")  # Label memory.
ax_a5[1].bar([str(x) for x in support_counts_a5], query_cost_a5, color="purple")  # Plot query evaluations.
ax_a5[1].set_title("prediction kernels")  # Title second panel.
ax_a5[1].set_xlabel("support count")  # Label x-axis.
plt.suptitle("Advanced 5: exact kernel scaling")  # Overall title.
plt.show()  # Display cost plots.

▶ What you'll see: training storage is quadratic in n, while prediction grows with the number of support examples.

👀 Takeaway: kernel methods are powerful, but exact Gram matrices and support-example prediction can become the bottleneck.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

A kernel computes inner products in a feature space without ever building that space explicitly.

Kernel methods make nonlinear structure look linear in an implicit feature space. The notebook checks the lesson's kernel arithmetic, then uses real RBF-kernel SVMs to show why the trick matters on nonlinear rungs. Save a copy to Drive to edit.

In [ ]:

import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
np.random.seed(7)

def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...].

    All X are 2-D float feature matrices, y integer labels, so one classifier runs unchanged
    across every rung (the 'watch it scale' story). Rungs get harder: clean+separable -> real
    high-dimensional. D1 is hand-built and fully inspectable.
    """
    rungs = []

    # D1 — four hand-placed 2-D points, 2 classes, clearly separable.
    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    # D2 — clean, well-separated Gaussian blobs.
    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    # D3 — non-linear, overlapping two-moons with noise.
    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    # D4 — real: Wine, 13 features, 3 classes.
    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    # D5 — real, harder: Breast Cancer, 30 features, class imbalance.
    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)



def _fit_predict(model, x_tr, y_tr, x_te):
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def _stump_adaboost(n_estimators=60, learning_rate=0.6):
    stump = DecisionTreeClassifier(max_depth=1, random_state=7)
    try:
        return AdaBoostClassifier(estimator=stump, n_estimators=n_estimators, learning_rate=learning_rate, random_state=7)
    except TypeError:
        return AdaBoostClassifier(base_estimator=stump, n_estimators=n_estimators, learning_rate=learning_rate, random_state=7)


def _project2d(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 1:
        return np.c_[X[:, 0], np.zeros(X.shape[0])]
    return X[:, :2]


def _plot_regions(ax, model, X, y, title):
    x2 = _project2d(X)
    scaler = StandardScaler()
    xs = scaler.fit_transform(x2)
    fitted = clone(model)
    try:
        fitted.fit(xs, y)
    except ValueError:
        fitted = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
        fitted.fit(xs, y)
    x_min = xs[:, 0].min() - 0.8
    x_max = xs[:, 0].max() + 0.8
    y_min = xs[:, 1].min() - 0.8
    y_max = xs[:, 1].max() + 0.8
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 80), np.linspace(y_min, y_max, 80))
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = fitted.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, cmap="tab10")
    ax.scatter(xs[:, 0], xs[:, 1], c=y, s=16, cmap="tab10", edgecolor="k", linewidth=0.2)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


def summarize_ladder(rungs):
    for name, X, y in rungs:
        classes, counts = np.unique(y, return_counts=True)
        print(f"{name}: X={X.shape}, classes={dict(zip(classes.tolist(), counts.tolist()))}")
    sample_name, sample_X, sample_y = rungs[0]
    print("sample rung:", sample_name)
    print(np.c_[sample_X, sample_y][:4])


def run_ladder(build_and_predict):
    rows = []
    for i, (name, X, y) in enumerate(clf_ladder(), start=1):
        acc = clf_accuracy(build_and_predict, X, y)
        rows.append((i, name, float(acc)))
    print("rung | accuracy | dataset")
    for i, name, acc in rows:
        print(f"D{i} | {acc:.3f} | {name}")
    return rows


def plot_summary(rows, model_factory, title):
    rungs = clf_ladder()
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    axes = axes.ravel()
    for ax, (name, X, y) in zip(axes[:5], rungs):
        _plot_regions(ax, model_factory(), X, y, name.split("(")[0])
    axes[5].plot([r[0] for r in rows], [r[2] for r in rows], marker="o")
    axes[5].set_ylim(0.0, 1.05)
    axes[5].set_xlabel("ladder rung")
    axes[5].set_ylabel("held-out accuracy")
    axes[5].set_title(title)
    axes[5].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def lesson_score(losses, cost, alternative):
    empirical = round(sum(losses) / len(losses), 3)
    score = round(empirical + cost, 3)
    gap = round(alternative - score, 3)
    return empirical, score, gap


def cost_sensitive_choice(raw_loss, cost, competitor):
    score = raw_loss + cost
    return score, score < competitor


## The concept, built once (D1)

The lesson formula is $$K(x,z)=\phi(x)^\top\phi(z)$$. We first rebuild the tiny score: average the three lesson losses, add the cost, and assert the exact plan numbers.

In [ ]:

def the_kernel_trick_kernel_methods_method():
    losses = np.array([0.213, 0.135, 0.471], dtype=float)
    empirical, score, gap = lesson_score(losses, 0.05, 0.359)
    assert empirical == 0.273
    assert score == 0.323
    assert gap == 0.036
    x = np.array([1.0, 2.0])
    z = np.array([3.0, -1.0])
    phi_x = np.array([x[0] ** 2, math.sqrt(2) * x[0] * x[1], x[1] ** 2])
    phi_z = np.array([z[0] ** 2, math.sqrt(2) * z[0] * z[1], z[1] ** 2])
    direct_kernel = float((x @ z) ** 2)
    feature_dot = float(phi_x @ phi_z)
    assert abs(direct_kernel - feature_dot) < 1e-12
    return {"empirical": empirical, "score": score, "gap": gap, "direct_kernel": direct_kernel, "feature_dot": feature_dot}

result = the_kernel_trick_kernel_methods_method()
print(result)


The printed dictionary contains the hand-checkable lesson score plus one method-specific quantity: a vote weight, an additive update, a second-order surrogate, a blend, a margin objective, or a kernel identity.

In [ ]:
checked = the_kernel_trick_kernel_methods_method()
assert checked['score'] == 0.323
print('D1 arithmetic verified for 3.30')

## The dataset ladder

All classification notebooks use the shared `clf_ladder()` and `clf_accuracy()` helpers embedded above, so the notebook is self-contained in Colab.

In [ ]:
rungs = clf_ladder()
summarize_ladder(rungs)

## Run the same method across D1-D5

The metric is held-out accuracy. Macro-F1 would be a useful companion when class skew is severe, especially on D5.

In [ ]:


def build_and_predict(x_tr, y_tr, x_te):
    model = SVC(kernel='rbf', C=2.0, gamma='scale', random_state=7)
    return _fit_predict(model, x_tr, y_tr, x_te)

rows = run_ladder(build_and_predict)
assert len(rows) == 5
assert all(0.0 <= acc <= 1.0 for _, _, acc in rows)


## Results visualization

The small multiples show the learned artifact on the first two standardized features for every rung; the summary panel tracks accuracy as the data become more realistic.

In [ ]:
plot_summary(rows, lambda: SVC(kernel='rbf', C=2.0, gamma='scale', random_state=7), 'The kernel trick & kernel methods accuracy')

## Pitfall on D5: optimizing the raw term and forgetting the cost

On D5, a flexible RBF kernel can overfit if gamma is too high. The lesson's fix is to compare on the same score scale and include cost/validation gap. The wrong behavior ranks by raw validation loss; the fix ranks by the lesson score with cost and the validation-gap scale.

In [ ]:

X, y = clf_ladder()[-1][1:]
x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
scaler = StandardScaler()
x_tr = scaler.fit_transform(x_tr)
x_te = scaler.transform(x_te)
lean = SVC(kernel='rbf', C=2.0, gamma='scale', random_state=7)
large = SVC(kernel='rbf', C=2.0, gamma='scale', random_state=7)
if hasattr(large, "set_params"):
    params = large.get_params()
    if "n_estimators" in params:
        large.set_params(n_estimators=min(params["n_estimators"] * 3, 240))
    if "max_iter" in params and params["max_iter"] > 0:
        large.set_params(max_iter=min(params["max_iter"] * 3, 240))
    if "C" in params:
        large.set_params(C=25.0)
    if "gamma" in params:
        large.set_params(gamma=3.0)
lean.fit(x_tr, y_tr)
large.fit(x_tr, y_tr)
lean_loss = 1.0 - accuracy_score(y_te, lean.predict(x_te))
large_loss = 1.0 - accuracy_score(y_te, large.predict(x_te))
wrong_pick = "large" if large_loss <= lean_loss else "lean"
lean_score, lean_ok = cost_sensitive_choice(lean_loss, 0.05, large_loss + 0.05 + 0.036)
large_score = large_loss + 0.05 + 0.036
fixed_pick = "lean" if lean_score <= large_score else "large"
print("raw losses:", {"lean": round(lean_loss, 3), "large": round(large_loss, 3)})
print("wrong raw-loss pick:", wrong_pick)
print("cost-aware scores:", {"lean": round(lean_score, 3), "large": round(large_score, 3)})
print("fixed pick:", fixed_pick)
assert lean_score <= large_score or fixed_pick == "large"


## Evaluate it + Practice

- Compare held-out accuracy against a majority-class no-skill baseline.
- Sanity check that shuffling labels pushes accuracy toward chance.
- Ablate the key idea: fewer boosting rounds, no honest stacking, linear instead of kernel, or tiny/huge C should change the metric.
- Watch failure signals: unstable D5 score, perfect train accuracy with weak validation accuracy, or a cost-aware score that reverses the raw-loss winner.

Practice 1: change one hyperparameter and re-run the D1-D5 table.

Practice 2: add a majority-class baseline row for every rung.

Practice 3: repeat D5 with a different random split and compare the validation gap.